# 14 · Reading and writing files (Spark)

Spark reads and writes many formats: CSV, JSON, **Parquet** (columnar, compressed, keeps types: the default
choice), ORC, Avro, and tables (Delta, Iceberg, BigQuery via a connector).

```
spark.read.format("csv").option("header", True).schema(...).load(path)
df.write.mode("overwrite").partitionBy("col").parquet(path)
```
A Spark "file" is really a **folder** with one part-file per partition. Practice files go to `tmp/` (not uploaded to git).

**How to run:** Start a cell with `%%sql` to write Spark SQL. In a normal Python cell use the DataFrame API: the tables are ready as variables (`employees`, `orders`, ...). Put your answer in `res` and call `res.show()`.

Tables: see [tables_guide.md](../../cheatsheets/tables_guide.md). Run the setup cell first (Shift+Enter).

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "spark"))
from spark_setup import get_spark, load_tables
from pyspark.sql import functions as F, Window, types as T

spark = get_spark()
t = load_tables(spark)          # also makes SQL views with the same names
employees, departments = t["employees"], t["departments"]
customers, orders = t["customers"], t["orders"]
products, order_items = t["products"], t["order_items"]
user_logins = t["user_logins"]
%load_ext spark_setup

## Q1. Read CSV: inferSchema vs explicit schema  ·  🟢 easy

Read `data/orders.tsv` (tab separated, header, NULL written as `NULL`) twice: (1) with `inferSchema=True`, (2) with an explicit schema. Print both schemas. What differs?

In [ ]:
# your answer here (DataFrame API)

## Q2. Bad rows: PERMISSIVE, DROPMALFORMED, FAILFAST  ·  🟡 medium

Write a small CSV with one bad row (`abc` in the amount column) to `tmp/bad.csv`. Read it with schema `id INT, amount DOUBLE` in the 3 modes and print the row count each time (FAILFAST will fail: catch the error). In PERMISSIVE mode keep the bad text in a `_corrupt` column.

In [ ]:
# your answer here (DataFrame API)

## Q3. Write and read Parquet  ·  🟢 easy

Write `orders` to `tmp/orders_pq` as Parquet (overwrite). Read it back, print the schema and the count. Then list the files in the folder.

In [ ]:
# your answer here (DataFrame API)

## Q4. Save modes  ·  🟢 easy

Write `departments` to `tmp/dept_pq` with mode `overwrite`, then twice with `append`. Print the count after each. Then try mode `errorifexists` and catch the error, and `ignore`.

In [ ]:
# your answer here (DataFrame API)

## Q5. Dynamic partition overwrite  ·  🔴 hard

Write orders partitioned by `order_month` (yyyy-MM text) to `tmp/orders_by_month`. Then rewrite **only** month `2025-10` with deduplicated rows, using `partitionOverwriteMode=dynamic`. Show that other months are untouched.

In [ ]:
# your answer here (DataFrame API)

## Q6. Export a pipe-delimited CSV + control file  ·  🔴 hard

Like a real hand-off to another system: write **completed** orders (deduplicated) as **one** CSV file with a header, `|` as separator, dates as `yyyy-MM-dd`, to `tmp/export`. Then write a control file `tmp/export.ctl` with the row count and the sum of amount. Print both files' first lines.

In [ ]:
# your answer here (DataFrame API)

## Q7. JSON: write, read, multiline  ·  🟡 medium

Write `products` as JSON lines to `tmp/products_json` and read it back. Then read a **multi-line** JSON array file `tmp/one.json` (write it first) with `multiLine=True`.

In [ ]:
# your answer here (DataFrame API)

## Q8. How many output files?  ·  🟢 easy

Write `order_items` with `repartition(3)` to `tmp/items_3` and with `coalesce(1)` to `tmp/items_1`. Count the part-files in each folder.

In [ ]:
# your answer here (DataFrame API)

## Q9. Schema evolution: mergeSchema  ·  🟡 medium

Write 2 Parquet folders under `tmp/evolve`: `v=1` with columns (id, name), `v=2` with (id, name, city). Read the parent folder with and without `mergeSchema` and compare the columns.

In [ ]:
# your answer here (DataFrame API)